# **Proyek Face Mask Detection menggunakan CNN**
Proyek ini bertujuan untuk melatih model Deep Learning (Convolutional Neural Network) untuk mendeteksi apakah seseorang menggunakan masker wajah atau tidak.

Dibuat oleh: **Fadhlur Rahman Fakhri**

### **1. Ekstraksi Dataset**

In [ ]:
# Hubungkan ke Google Drive jika diperlukan, atau extract zip dataset langsung
import zipfile
import os

zip_path = "/content/face-mask-detection.zip"
extract_path = "/content/dataset"

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Ekstraksi dataset berhasil!")
else:
    print("File zip dataset tidak ditemukan di /content/. Silakan upload dataset zip terlebih dahulu.")

In [ ]:
# Cek jumlah gambar tiap kelas
with_mask_dir = "/content/dataset/with mask"
without_mask_dir = "/content/dataset/without mask"

if os.path.exists(with_mask_dir) and os.path.exists(without_mask_dir):
    print("Jumlah gambar dengan masker:", len(os.listdir(with_mask_dir)))
    print("Jumlah gambar tanpa masker:", len(os.listdir(without_mask_dir)))
else:
    print("Direktori dataset tidak ditemukan. Pastikan path ekstraksi sudah benar.")

### **2. Prapemrosesan Data (Preprocessing dengan Face Cropping)**
Agar model memiliki akurasi yang tinggi saat dideploy (di mana wajah dideteksi terlebih dahulu lalu dipotong), kita harus melatih model pada **area wajah yang sudah dipotong (face crop)**, bukan gambar utuh yang menyertakan latar belakang.

In [ ]:
import cv2
import numpy as np
import os

# Load Haar Cascade untuk mendeteksi area wajah
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

data = []
labels = []

def load_and_preprocess_images(folder_path, label_value):
    count = 0
    for img_name in os.listdir(folder_path):
        img_path = os.path.join(folder_path, img_name)
        img = cv2.imread(img_path)
        if img is None:
            continue
        
        # Deteksi wajah menggunakan Haar Cascade
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(
            gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60)
        )
        
        # Jika wajah terdeteksi, potong area wajahnya
        if len(faces) > 0:
            x, y, w, h = faces[0]
            face = img[y:y+h, x:x+w]
        else:
            # Fallback jika wajah tidak terdeteksi (opsional, gunakan gambar utuh)
            face = img
            
        # Resize ke ukuran standard model 224x224
        face = cv2.resize(face, (224, 224))
        
        # Normalisasi pixel ke skala [0, 1]
        face = face.astype("float32") / 255.0
        
        data.append(face)
        labels.append(label_value)
        count += 1
    print(f"Berhasil memproses {count} gambar dari folder {os.path.basename(folder_path)}")

# Label 0: without mask, Label 1: with mask
load_and_preprocess_images(without_mask_dir, 0)
load_and_preprocess_images(with_mask_dir, 1)

data = np.array(data, dtype="float32")
labels = np.array(labels, dtype="int32")

print("Shape data latihan:", data.shape)
print("Shape label latihan:", labels.shape)

### **3. Pembagian Dataset (Split Data Training & Testing)**

In [ ]:
from sklearn.model_selection import train_test_split

# Split data 80% training dan 20% testing dengan stratifikasi agar seimbang
X_train, X_test, y_train, y_test = train_test_split(
    data, labels, test_size=0.2, random_state=42, stratify=labels
)

print("Jumlah data training :", X_train.shape[0])
print("Jumlah data testing  :", X_test.shape[0])
print("Training 'with mask'   :", sum(y_train == 1))
print("Training 'without mask':", sum(y_train == 0))

### **4. Membangun Model CNN**
Kami memperdalam arsitektur CNN dengan 4 layer konvolusi dan pooling untuk menekan overfitting dan menjaga parameter model tetap kecil (~3 juta parameter daripada model sebelumnya yang mencapai 24 juta parameter).

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential([
    # Layer 1
    Conv2D(32, (3,3), padding='same', activation='relu', input_shape=(224,224,3)),
    MaxPooling2D(pool_size=(2,2)),
    
    # Layer 2
    Conv2D(64, (3,3), padding='same', activation='relu'),
    MaxPooling2D(pool_size=(2,2)),
    
    # Layer 3
    Conv2D(128, (3,3), padding='same', activation='relu'),
    MaxPooling2D(pool_size=(2,2)),
    
    # Layer 4
    Conv2D(128, (3,3), padding='same', activation='relu'),
    MaxPooling2D(pool_size=(2,2)),
    
    # Fully Connected Layer
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid') # Output sigmoid untuk klasifikasi biner
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

### **5. Pelatihan Model (Training)**

In [ ]:
# Melatih model selama 10 epoch dengan validation set dari data test
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test)
)

In [ ]:
# Simpan model terlatih
model.save("face_mask_cnn.h5")
print("Model disimpan sebagai 'face_mask_cnn.h5'")

### **6. Evaluasi Grafik Training**

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

# Plot Akurasi
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Akurasi Pelatihan vs Validasi')
plt.xlabel('Epoch')
plt.ylabel('Akurasi')
plt.legend()

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss Pelatihan vs Validasi')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

### **7. Evaluasi Model (Confusion Matrix & Classification Report)**
Kami mengevaluasi performa model dengan test set. Pastikan Anda **TIDAK** menjalankan ulang sel pembuatan model (`model = Sequential(...)`) setelah training selesai agar bobot model tidak ter-reset!

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Jalankan evaluasi dasar
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print("Loss Terhadap Data Test:", loss)
print("Akurasi Terhadap Data Test: {:.2f}%".format(accuracy * 100))

# Dapatkan probabilitas prediksi untuk data test
predictions = model.predict(X_test)

# Konversi probabilitas sigmoid ke kelas biner (0 atau 1) dengan ambang batas 0.5
y_pred = (predictions >= 0.5).astype("int32")

# Tampilkan Classification Report
# Pengurutan target_names harus sesuai label: 0 adalah 'without mask', 1 adalah 'with mask'
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['without mask', 'with mask']))

# Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=['without mask', 'with mask'],
    yticklabels=['without mask', 'with mask']
)
plt.title('Confusion Matrix')
plt.ylabel('Aktual')
plt.xlabel('Prediksi')
plt.show()

### **8. Konversi Model ke TensorFlow.js**
Kami mengekspor model menjadi format `layers-model` yang kompatibel dengan `tf.loadLayersModel` di browser JavaScript.

In [ ]:
!pip install tensorflowjs
!tensorflowjs_converter --input_format=keras "/content/face_mask_cnn.h5" "/content/tfjs_model"
print("Model berhasil dikonversi ke format TensorFlow.js dan disimpan di folder /content/tfjs_model")

### **9. Pengujian Prediksi Foto Wajah Baru**

In [ ]:
from google.colab import files
uploaded = files.upload()

if len(uploaded) > 0:
    filename = list(uploaded.keys())[0]
    img = cv2.imread(filename)
    img_show = img.copy()
    
    # Deteksi Wajah
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)
    
    if len(faces) > 0:
        # Proses wajah pertama
        x, y, w, h = faces[0]
        face = img[y:y+h, x:x+w]
        face = cv2.resize(face, (224, 224))
        face = face.astype("float32") / 255.0
        face = np.expand_dims(face, axis=0)
        
        # Prediksi
        prediction = model.predict(face)
        confidence = prediction[0][0]
        
        # Ambil label & warna
        # val >= 0.5 => with mask (Label 1)
        # val < 0.5 => without mask (Label 0)
        is_mask = confidence >= 0.5
        label = "With Mask" if is_mask else "Without Mask"
        color = (0, 255, 0) if is_mask else (0, 0, 255)
        
        score = confidence if is_mask else (1.0 - confidence)
        text = f"{label} ({score*100:.2f}%)"
        
        # Gambar Box
        cv2.rectangle(img_show, (x, y), (x+w, y+h), color, 2)
        cv2.putText(img_show, text, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
        
        # Tampilkan Hasil
        plt.figure(figsize=(7, 7))
        plt.imshow(cv2.cvtColor(img_show, cv2.COLOR_BGR2RGB))
        plt.axis("off")
        plt.title("Hasil Deteksi Masker Wajah")
        plt.show()
    else:
        print("Tidak ada wajah yang terdeteksi pada gambar.")